# AccentCat — StarGANv2-VC Real-time Accent Conversion Server


## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 2 — Install dependencies

⚠️ **Important Setup Notice**

To ensure compatibility with the voice conversion models (specifically `parallel_wavegan`), this notebook requires downgrading certain core libraries (like NumPy to version 1.26.4).

**What to expect when you run the setup cell below:**
1. It will download and install the required dependencies.
2. Upon completion, the script will **intentionally crash and restart** the runtime. You might see a Colab warning saying *"Your session crashed for an unknown reason."* 3. **Do not panic!** This is expected behavior to force Python to load the newly installed library versions.
4. Just wait 2-3 seconds for it to reconnect, and then you can safely proceed to the next cells. You do not need to run the setup cell a second time.

In [2]:
import os

# Check if the environment has already been configured
if not os.path.exists('/content/.env_ready'):
    print("🚀 First-time setup: Installing dependencies. The session will restart automatically upon completion...")

    # 1. Install system dependencies (ffmpeg for audio processing)
    !apt-get install -y -qq ffmpeg

    # 2. Install core Python packages
    !pip install -q fastapi uvicorn websockets pydub soundfile pyngrok nest-asyncio munch parallel_wavegan
    !pip install -q scipy==1.11.4

    # 3. Force install specific numpy version (ignores dependency warnings from newer Colab pre-installs)
    !pip install -q --no-build-isolation numpy==1.26.4 --force-reinstall

    # Mark setup as complete
    with open('/content/.env_ready', 'w') as f:
        f.write('done')

    print("✅ Setup complete. Restarting runtime to apply changes...")

    # 4. Force restart the Colab runtime
    os.kill(os.getpid(), 9)

else:
    # If the cell is run again after the restart, it will just verify the versions
    import numpy as np
    import scipy
    print("✨ Environment is ready! No restart needed.")
    print(f"📦 Active NumPy version: {np.__version__}")
    print(f"📦 Active SciPy version: {scipy.__version__}")

✨ Environment is ready! No restart needed.
📦 Active NumPy version: 1.26.4
📦 Active SciPy version: 1.11.4


In [3]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 129.2 MB/s eta 0:00:00


## Cell 3 — Configuration

In [13]:
# ==============================================================
# USER CONFIGURATION
# ==============================================================
WORK_DIR        = '/content/drive/MyDrive/Github/Accent-Conversion'
CHECKPOINT_PATH = 'Models/VCTK2019/epoch_00150.pth'
CONFIG_YAML     = 'Models/VCTK2019/config.yml'

# Vocoder paths — confirmed from your Vocoder/ directory listing
VOCODER_CONFIG  = 'Vocoder/config.yml'
VOCODER_CKPT    = 'Vocoder/checkpoint-400000steps.pkl'

# Fixed reference speaker for the demo (Data/p225/1.wav ~ 10.wav)
DEMO_REFS = {
    'p228': 'Data/VCTK2019/p228/23.wav',
    'p230': 'Data/VCTK2019/p230/23.wav',
    'p233': 'Data/VCTK2019/p233/23.wav',
    'p236': 'Data/VCTK2019/p236/23.wav',
    'p243': 'Data/VCTK2019/p243/23.wav',
    'p244': 'Data/VCTK2019/p244/23.wav',
    'p254': 'Data/VCTK2019/p254/23.wav',
    'p258': 'Data/VCTK2019/p258/23.wav',
    'p259': 'Data/VCTK2019/p259/23.wav',
    'p273': 'Data/VCTK2019/p273/23.wav',
}

NGROK_AUTH_TOKEN = '3BUOT4U5ZAcooHpRtdJlypPVZmo_5YiZj2TkVKS8A2WpkcNKV'
NGROK_DOMAIN     = 'indistinctively-nonhesitant-briggs.ngrok-free.dev'
SERVER_PORT      = 8000

print('Configuration set.')

Configuration set.


## Cell 4 — Load all models

In [15]:
import sys, os, io, yaml, torch, librosa
import subprocess
import numpy as np
import soundfile as sf
import torchaudio
from munch import Munch

sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_NAME = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
print(f'Using device: {device} ({DEVICE_NAME})')

# ------------------------------------------------------------------
# Stage 1: Import project modules
# ------------------------------------------------------------------
print('[1] Importing project modules...')
from src.models import build_model
from Utils.JDC.model import JDCNet
from Utils.ASR.models import ASRCNN
from parallel_wavegan.utils import load_model as load_vocoder
print('  OK')

# ------------------------------------------------------------------
# Stage 2: Parse training config
# ------------------------------------------------------------------
print('[2] Loading config...')
with open(os.path.join(WORK_DIR, CONFIG_YAML)) as f:
    config = yaml.safe_load(f)

# ------------------------------------------------------------------
# Stage 3: Build speaker -> domain index map from train_list.txt
# ------------------------------------------------------------------
print('[3] Building speaker->domain map from train_list.txt...')
SPEAKER_DOMAIN_MAP = {}
train_list_path = os.path.join(WORK_DIR, config.get('train_data', 'Data/VCTK2019/train_list.txt'))
with open(train_list_path) as f:
    for line in f:
        line = line.strip()
        if '|' not in line:
            continue
        path_part, domain_str = line.rsplit('|', 1)
        # Extract speaker id: './Data/p225/22.wav' -> 'p225'
        parts = path_part.replace('\\', '/').split('/')
        # Find the segment that looks like a speaker id (starts with 'p')
        speaker = None
        for seg in parts:
            if seg.startswith('p') and seg[1:].isdigit():
                speaker = seg
                break
        if speaker and speaker not in SPEAKER_DOMAIN_MAP:
            SPEAKER_DOMAIN_MAP[speaker] = int(domain_str)
print(f'  Found {len(SPEAKER_DOMAIN_MAP)} speakers:')
for spk, dom in sorted(SPEAKER_DOMAIN_MAP.items()):
    print(f'    {spk} -> domain {dom}')



# ------------------------------------------------------------------
# Stage 4: Load F0 model (JDCNet)
# ------------------------------------------------------------------
print('[4] Loading F0 (JDC) model...')
F0_path  = config.get('F0_path', 'Utils/JDC/bst.t7')
F0_model = JDCNet(num_class=1, seq_len=192)
f0_ckpt  = torch.load(os.path.join(WORK_DIR, F0_path), map_location=device)
F0_model.load_state_dict(f0_ckpt['net'])
F0_model = F0_model.to(device).eval()
print('  F0 ready')

# ------------------------------------------------------------------
# Stage 5: Load ASR model
# ------------------------------------------------------------------
print('[5] Loading ASR model...')
ASR_config_path = config.get('ASR_config', 'Utils/ASR/config.yml')
ASR_model_path  = config.get('ASR_path',   'Utils/ASR/epoch_00100.pth')
with open(os.path.join(WORK_DIR, ASR_config_path)) as f:
    asr_cfg = yaml.safe_load(f)
ASR_model = ASRCNN(**asr_cfg['model_params'])
asr_ckpt  = torch.load(os.path.join(WORK_DIR, ASR_model_path),
                        map_location=device, weights_only=False)['model']
ASR_model.load_state_dict(asr_ckpt)
ASR_model = ASR_model.to(device).eval()
print('  ASR ready')

# ------------------------------------------------------------------
# Stage 6: Build StarGANv2-VC and load checkpoint
# ------------------------------------------------------------------
print('[6] Building StarGANv2-VC...')
args = Munch(config['model_params'])
nets, nets_ema = build_model(args, F0_model, ASR_model)

print('[7] Loading checkpoint...')
ckpt = torch.load(os.path.join(WORK_DIR, CHECKPOINT_PATH),
                  map_location=device, weights_only=False)
# Support both flat and nested checkpoint formats
if 'model' in ckpt:
    model_dict = ckpt['model']
    # Try common key variations
    gen_sd   = (model_dict.get('generator')
                or model_dict.get('Generator')
                or model_dict.get('gen'))
    style_sd = (model_dict.get('style_encoder')
                or model_dict.get('StyleEncoder')
                or model_dict.get('style'))
else:
    gen_sd   = ckpt.get('generator') or ckpt.get('Generator')
    style_sd = ckpt.get('style_encoder') or ckpt.get('StyleEncoder')

if gen_sd is None or style_sd is None:
    print('Available checkpoint keys:', list(ckpt.keys()))
    if 'model' in ckpt:
        print('Keys inside ckpt[model]:', list(ckpt['model'].keys()))
    raise RuntimeError('Cannot find generator/style_encoder in checkpoint. See keys above.')

nets_ema.generator.load_state_dict(gen_sd)
nets_ema.style_encoder.load_state_dict(style_sd)
for m in nets_ema.values():
    m.to(device).eval()
print('  StarGAN ready')

# ------------------------------------------------------------------
# Stage 7: Load vocoder
# Note: config.yml does NOT have vocoder keys, so we use the
# explicit VOCODER_CONFIG / VOCODER_CKPT variables set in Cell 3.
# ------------------------------------------------------------------
print('[8] Loading vocoder...')
print(f'  ckpt : {VOCODER_CKPT}')
print(f'  cfg  : {VOCODER_CONFIG}')

vocoder_ckpt_path   = os.path.join(WORK_DIR, VOCODER_CKPT)
vocoder_config_path = os.path.join(WORK_DIR, VOCODER_CONFIG)

if not os.path.exists(vocoder_ckpt_path):
    raise FileNotFoundError(f'Vocoder checkpoint not found: {vocoder_ckpt_path}')
if not os.path.exists(vocoder_config_path):
    raise FileNotFoundError(f'Vocoder config not found: {vocoder_config_path}')

vocoder = load_vocoder(vocoder_ckpt_path)
vocoder.to(device).eval()
print('  Vocoder ready')
print('\n✅ All models loaded successfully!')

Using device: cuda (NVIDIA L4)
[1] Importing project modules...
  OK
[2] Loading config...
[3] Building speaker->domain map from train_list.txt...
  Found 20 speakers:
    p225 -> domain 0
    p226 -> domain 10
    p227 -> domain 11
    p228 -> domain 1
    p229 -> domain 2
    p230 -> domain 3
    p231 -> domain 4
    p232 -> domain 12
    p233 -> domain 5
    p236 -> domain 6
    p239 -> domain 7
    p240 -> domain 8
    p243 -> domain 13
    p244 -> domain 9
    p254 -> domain 14
    p256 -> domain 15
    p258 -> domain 16
    p259 -> domain 17
    p270 -> domain 18
    p273 -> domain 19
[4] Loading F0 (JDC) model...
  F0 ready
[5] Loading ASR model...
  ASR ready
[6] Building StarGANv2-VC...
[7] Loading checkpoint...
  StarGAN ready
[8] Loading vocoder...
  ckpt : Vocoder/checkpoint-400000steps.pkl
  cfg  : Vocoder/config.yml


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Vocoder ready

✅ All models loaded successfully!


## Cell 5 — Mel / inference helpers

In [16]:
import torch.nn.functional as F

# ── Constants ──────────────────────────────────────────────────────────
SAMPLE_RATE = 24000
N_MELS      = 80        # Fixed to 80, do not use dim_in
N_FFT       = 2048
WIN_LENGTH  = 1200
HOP_LENGTH  = 300
MEL_MEAN    = -4        # Normalization parameter used during training
MEL_STD     =  4

MIN_MEL_FRAMES    = 64
VAD_RMS_THRESHOLD = 0.005

to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=N_MELS, n_fft=N_FFT,
    win_length=WIN_LENGTH, hop_length=HOP_LENGTH
).to(device)

def compute_mel(wav_tensor):
    """wav_tensor: (T,) or (1,T) float32  →  (1, 80, T_mel)"""
    if wav_tensor.dim() == 1:
        wav_tensor = wav_tensor.unsqueeze(0)
    mel = to_mel(wav_tensor)                               # (1, 80, T)
    mel = (torch.log(1e-5 + mel) - MEL_MEAN) / MEL_STD   # Normalization, matching the training phase
    return mel

def load_ref_wav(wav_path: str) -> torch.Tensor:
    wave, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
    wave, _  = librosa.effects.trim(wave, top_db=30)      # Trim leading and trailing silence
    wave     = wave / (np.max(np.abs(wave)) + 1e-9)       # Normalize amplitude
    t = torch.FloatTensor(wave).to(device)
    return compute_mel(t)   # (1, 80, T)

def decode_webm_to_pcm(webm_bytes: bytes) -> np.ndarray:
    import subprocess, tempfile
    with tempfile.NamedTemporaryFile(suffix='.webm', delete=False) as fin:
        fin.write(webm_bytes)
        in_path = fin.name
    out_path = in_path.replace('.webm', '.wav')
    try:
        r = subprocess.run(
            ['ffmpeg', '-y', '-i', in_path,
             '-ar', str(SAMPLE_RATE), '-ac', '1', '-sample_fmt', 's16', out_path],
            capture_output=True, timeout=30
        )
        if r.returncode != 0:
            raise RuntimeError('ffmpeg: ' + r.stderr.decode(errors='replace')[-400:])
        samples, _ = sf.read(out_path, dtype='float32')
        return samples
    finally:
        for p in [in_path, out_path]:
            try: os.remove(p)
            except: pass

def is_silence(samples: np.ndarray) -> bool:
    rms = float(np.sqrt(np.mean(samples ** 2)))
    print(f'  [VAD] RMS={rms:.5f}  threshold={VAD_RMS_THRESHOLD}')
    return rms < VAD_RMS_THRESHOLD

def run_inference(samples: np.ndarray,
                  ref_mel: torch.Tensor,
                  domain_idx: int) -> np.ndarray:
    # Normalize source audio amplitude, consistent with inference.ipynb
    samples = samples / (np.max(np.abs(samples)) + 1e-9)

    wav_t   = torch.FloatTensor(samples).to(device)
    src_mel = compute_mel(wav_t)   # (1, 80, T)

    # Pad short audio clips
    if src_mel.shape[-1] < MIN_MEL_FRAMES:
        pad = MIN_MEL_FRAMES - src_mel.shape[-1]
        src_mel = F.pad(src_mel, (0, pad))
        print(f'  [PAD] padded {pad} frames')

    src_mel_4d = src_mel.unsqueeze(1)    # (1, 1, 80, T)
    ref_mel_4d = ref_mel.unsqueeze(1)    # (1, 1, 80, T_ref)
    domain_t   = torch.LongTensor([domain_idx]).to(device)

    with torch.no_grad():
        # Style vector
        style_vec = nets_ema.style_encoder(ref_mel_4d, domain_t)

        # F0 features, extracted using get_feature_GAN, consistent with inference.ipynb
        f0_feat = F0_model.get_feature_GAN(src_mel_4d)

        # Generator
        conv_mel = nets_ema.generator(src_mel_4d, style_vec, F0=f0_feat)

        # Vocoder expects shape (T, 80)
        c = conv_mel.transpose(-1, -2).squeeze().to(device)
        wav_out = vocoder.inference(c)
        wav_out = wav_out.view(-1).detach().to('cpu').detach().cpu().numpy().astype(np.float32)

    # Clear NaN/Inf values and normalize
    wav_out = np.nan_to_num(wav_out, nan=0.0, posinf=0.0, neginf=0.0)
    peak = np.max(np.abs(wav_out))
    if peak > 1e-6:
        wav_out = (wav_out / peak) * 0.95
    else:
        print('  [WARN] Near-silent output')
    return wav_out

print(f'Helpers defined. SR={SAMPLE_RATE}  N_MELS={N_MELS}  MEL_MEAN={MEL_MEAN}  MEL_STD={MEL_STD}')
print(f'Inference device: {device}')

Helpers defined. SR=24000  N_MELS=80  MEL_MEAN=-4  MEL_STD=4
Inference device: cuda


In [17]:
import os
from IPython.display import Audio, display

def backend_record_and_convert(
    target_speakers=None,
    filename='my_test.wav',
    output_dir='multi_converted_outputs',
    sample_rate=24000,
    play_audio=True
):
    """
    Record once, then convert to multiple target speakers
    """
    if target_speakers is None:
        target_speakers = [
            'p228', 'p230', 'p233', 'p236', 'p243',
            'p244', 'p254', 'p258', 'p259', 'p273'
        ]

    os.makedirs(output_dir, exist_ok=True)

    # 1) preload references
    refs = ensure_preloaded_refs()

    # 2) record once
    print('=' * 60)
    print('Recording source audio once...')
    source_wav = record_audio_notebook(filename=filename, sample_rate=sample_rate)
    print(f'[OK] Source recorded: {source_wav}')

    if play_audio:
        print('\n[Source Audio]')
        display(Audio(source_wav))

    # 3) convert for each target speaker
    results = {}

    for spk in target_speakers:
        print('\n' + '=' * 60)
        print(f'Target speaker: {spk}')

        if spk not in refs:
            print(f'[Skip] {spk} not found in preloaded refs')
            continue

        try:
            ref_mel, ref_domain = refs[spk]

            converted_wav = run_inference(
                source_wav,
                ref_mel,
                ref_domain
            )

            out_path = os.path.join(output_dir, f'converted_{spk}.wav')
            save_wav(out_path, converted_wav, sr=sample_rate)

            results[spk] = out_path
            print(f'[OK] Saved: {out_path}')

            if play_audio:
                print(f'[Converted -> {spk}]')
                display(Audio(out_path))

        except Exception as e:
            print(f'[Failed] {spk}: {e}')

    print('\n' + '=' * 60)
    print('[DONE] All conversions finished.')
    print('Generated files:')
    for spk, path in results.items():
        print(f'  {spk}: {path}')

    return results


## Cell 6 — Notebook backend recording demo



In [18]:

# =========================
# Notebook backend demo helpers
# =========================
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import IPython.display as ipd
import matplotlib.pyplot as plt
import tempfile

NOTEBOOK_RECORD_DIR = os.path.join(WORK_DIR, 'Demo/live_inputs')
NOTEBOOK_OUTPUT_DIR = os.path.join(WORK_DIR, 'Demo/converted_results')
os.makedirs(NOTEBOOK_RECORD_DIR, exist_ok=True)
os.makedirs(NOTEBOOK_OUTPUT_DIR, exist_ok=True)

PRELOADED_REFS = globals().get('PRELOADED_REFS', {})

def ensure_preloaded_refs():
    """
    Load all demo reference mels once.
    Returns:
        dict: {speaker_id: (ref_mel, domain_idx)}
    """
    global PRELOADED_REFS
    if PRELOADED_REFS:
        print(f'Using cached reference mels: {list(PRELOADED_REFS.keys())}')
        return PRELOADED_REFS

    print('Pre-loading reference mels for notebook demo...')
    loaded = {}
    for spk, rel_path in DEMO_REFS.items():
        wav_path = os.path.join(WORK_DIR, rel_path)
        if not os.path.exists(wav_path):
            print(f'  ⚠️ Missing: {rel_path}')
            continue

        mel = load_ref_wav(wav_path)
        domain = SPEAKER_DOMAIN_MAP.get(spk, 0)
        loaded[spk] = (mel, domain)
        print(f'  ✅ {spk} (domain={domain}) mel={tuple(mel.shape)}')

    if not loaded:
        raise RuntimeError('No reference clips found for notebook demo.')

    PRELOADED_REFS = loaded
    print('Reference preload done.')
    return PRELOADED_REFS

def record_audio_notebook(filename='live_input.wav', sample_rate=SAMPLE_RATE):
    """
    Record audio in notebook browser, save as wav, and return wav path.
    """
    js = Javascript("""
    async function recordAudio() {
      const wrap = document.createElement('div');
      wrap.style = "padding:8px 8px;";
      const startButton = document.createElement('button');
      const stopButton = document.createElement('button');
      startButton.textContent = 'Start Recording';
      stopButton.textContent = 'Stop Recording';
      startButton.style.marginRight = '8px';
      wrap.appendChild(startButton);
      wrap.appendChild(stopButton);
      document.body.appendChild(wrap);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      const recorded = new Promise(resolve => {
        recorder.onstop = async () => {
          const blob = new Blob(chunks, {type: 'audio/webm'});
          const arrayBuffer = await blob.arrayBuffer();
          const uint8Array = new Uint8Array(arrayBuffer);
          let binary = '';
          uint8Array.forEach(b => binary += String.fromCharCode(b));
          stream.getTracks().forEach(track => track.stop());
          wrap.remove();
          resolve(btoa(binary));
        };
      });

      startButton.onclick = () => recorder.start();
      stopButton.onclick = () => recorder.stop();

      return await recorded;
    }
    """)
    display(js)
    audio_data = output.eval_js("recordAudio()")
    audio_bytes = b64decode(audio_data)

    webm_path = os.path.join(NOTEBOOK_RECORD_DIR, 'temp_record.webm')
    wav_path  = os.path.join(NOTEBOOK_RECORD_DIR, filename)

    with open(webm_path, 'wb') as f:
        f.write(audio_bytes)

    subprocess.run([
        'ffmpeg', '-y',
        '-i', webm_path,
        '-ar', str(sample_rate),
        '-ac', '1',
        wav_path
    ], check=True)

    return wav_path

def load_source_audio(wav_path):
    audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    audio = audio / (np.max(np.abs(audio)) + 1e-9)
    return audio.astype(np.float32)

def tensor_to_mel_numpy(mel_tensor):
    if isinstance(mel_tensor, torch.Tensor):
        mel = mel_tensor.detach().cuda()
        if mel.dim() == 4:
            mel = mel.squeeze(0).squeeze(0)
        elif mel.dim() == 3:
            mel = mel.squeeze(0)
        mel = mel.detach().cpu().numpy()
    else:
        mel = mel_tensor
    return mel

def show_mel(mel_tensor, title='Mel Spectrogram'):
    mel = tensor_to_mel_numpy(mel_tensor)
    plt.figure(figsize=(10, 4))
    plt.imshow(mel, aspect='auto', origin='lower')
    plt.title(title)
    plt.xlabel('Frames')
    plt.ylabel('Mel bins')
    plt.colorbar()
    plt.tight_layout()
    plt.show()

def preprocess_wav_and_mel(wav_path):
    """
    Loads a wav file, normalizes it, and computes its mel spectrogram.
    Returns: (np.ndarray, torch.Tensor) or (None, None) if silence
    """
    wav = load_source_audio(wav_path)

    if is_silence(wav):
        print(f'[VAD] Detected silence in source audio. Skipping inference.')
        return None, None

    # Convert numpy array to torch tensor for mel computation
    wav_tensor = torch.FloatTensor(wav).to(device)
    mel = compute_mel(wav_tensor)
    return wav, mel

def compute_mel_np(wav_np, sr):
    """
    Computes mel spectrogram from a numpy array.
    """
    wav_tensor = torch.FloatTensor(wav_np).to(device)
    mel = to_mel(wav_tensor).detach().cuda().numpy()
    mel = (np.log(1e-5 + mel) - MEL_MEAN) / MEL_STD
    return mel

def backend_record_and_convert(
    target_speakers=None,
    filename='live_input.wav',
    output_dir='multi_converted_outputs',
    sample_rate=24000,
    play_audio=True,
    show_plots=True
):
    """
    Record once in notebook and convert to multiple target speakers using
    the same backend inference pipeline.

    Args:
        target_speakers (list[str] | None): target speakers to convert to.
        filename (str): recorded wav filename.
        output_dir (str): folder to save converted wavs.
        sample_rate (int): recording sample rate.
        play_audio (bool): whether to play source/converted audios.
        show_plots (bool): whether to show source/reference/converted mels.
    """
    if target_speakers is None:
        target_speakers = ['p228', 'p230', 'p233', 'p236', 'p243',
                           'p244', 'p254', 'p258', 'p259', 'p273']

    refs = ensure_preloaded_refs()
    valid_targets = [spk for spk in target_speakers if spk in refs]
    if not valid_targets:
        raise RuntimeError('No valid target speakers found in preloaded refs.')

    os.makedirs(output_dir, exist_ok=True)

    print('=' * 60)
    print('Recording source audio once...')
    source_wav_path = record_audio_notebook(filename=filename, sample_rate=sample_rate)
    print(f'[OK] Source recorded: {source_wav_path}')

    wav, src_mel = preprocess_wav_and_mel(source_wav_path)
    if wav is None:
        raise RuntimeError('Failed to preprocess recorded audio.')

    if play_audio:
        print('\n[Source Audio]')
        display(ipd.Audio(source_wav_path))

    results = {}

    for spk in valid_targets:
        print('\n' + '=' * 60)
        print(f'Target speaker: {spk}')

        ref_mel, ref_domain = refs[spk]
        ref_path = DEMO_REFS[spk]
        ref_full_path = resolve_demo_ref_path(ref_path)

        try:
            converted_np = run_inference(wav, ref_mel, ref_domain)
            converted_np = np.asarray(converted_np, dtype=np.float32)

            out_path = os.path.join(output_dir, f'converted_{spk}.wav')
            sf.write(out_path, converted_np, 24000)
            results[spk] = out_path
            print(f'[OK] Saved: {out_path}')

            if play_audio:
                print(f'[Converted -> {spk}]')
                display(ipd.Audio(out_path))

            if show_plots:
                converted_mel = compute_mel_np(converted_np, sr=24000)

                fig = plt.figure(figsize=(15, 4))
                plt.subplot(1, 3, 1)
                plt.imshow(src_mel.squeeze().cuda().detach().cpu().numpy(), aspect='auto', origin='lower')
                plt.title('Source mel')
                plt.xlabel('Frames')
                plt.ylabel('Mel bins')

                plt.subplot(1, 3, 2)
                plt.imshow(ref_mel.squeeze().cuda().detach().cpu().numpy(), aspect='auto', origin='lower')
                plt.title(f'Reference mel ({spk})')
                plt.xlabel('Frames')

                plt.subplot(1, 3, 3)
                plt.imshow(converted_mel, aspect='auto', origin='lower')
                plt.title(f'Converted mel ({spk})')
                plt.xlabel('Frames')

                plt.tight_layout()
                plt.show()

        except Exception as e:
            print(f'[Failed] {spk}: {e}')

    print('\n' + '=' * 60)
    print('[DONE] All conversions finished.')
    print('Generated files:')
    for spk, out_path in results.items():
        print(f'  {spk}: {out_path}')

    return results


## Cell 7 — Run notebook recording demo

```python
ensure_preloaded_refs()
backend_record_and_convert(
    target_speakers=['p228', 'p230', 'p233', 'p236', 'p243', 'p244', 'p254', 'p258', 'p259', 'p273'],
    filename='my_test.wav'
)
```


In [19]:
PRELOADED_REFS = globals().get('PRELOADED_REFS', {})

def resolve_demo_ref_path(rel_path):
    """
    Resolve reference wav path using the same layout as inference notebook.
    """
    candidates = []

    norm_rel = rel_path.lstrip('/').replace('\\', '/')

    # 1) direct path under WORK_DIR
    candidates.append(os.path.join(WORK_DIR, norm_rel))

    # 2) if already starts with Demo/, also try stripping Demo/
    if norm_rel.startswith('Demo/'):
        candidates.append(os.path.join(WORK_DIR, norm_rel[len('Demo/'):]))
    else:
        # 3) inference-style path
        candidates.append(os.path.join(WORK_DIR, 'Demo', norm_rel))

    # 4) fallback old style
    speaker_folder = os.path.basename(os.path.dirname(norm_rel))
    wav_name = os.path.basename(norm_rel)
    candidates.append(os.path.join(WORK_DIR, 'VCTK-corpus', speaker_folder, wav_name))

    for c in candidates:
        if os.path.exists(c):
            return c

    return None


def ensure_preloaded_refs():
    """
    Load all demo reference mels once.
    Returns:
        dict: {speaker_id: (ref_mel, domain_idx)}
    """
    global PRELOADED_REFS
    if PRELOADED_REFS:
        print(f'Using cached reference mels: {list(PRELOADED_REFS.keys())}')
        return PRELOADED_REFS

    print('Pre-loading reference mels for notebook demo...')
    loaded = {}

    for spk, rel_path in DEMO_REFS.items():
        wav_path = resolve_demo_ref_path(rel_path)
        if wav_path is None:
            print(f'  ⚠️ Missing: {rel_path}')
            continue

        print(f'  📎 {spk}: {wav_path}')
        mel = load_ref_wav(wav_path)
        domain = SPEAKER_DOMAIN_MAP.get(spk, 0)
        loaded[spk] = (mel, domain)
        print(f'  ✅ {spk} (domain={domain}) mel={tuple(mel.shape)}')

    if not loaded:
        raise RuntimeError('No reference clips found for notebook demo.')

    PRELOADED_REFS = loaded
    print('Reference preload done.')
    return PRELOADED_REFS

In [20]:

!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null


!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list


!sudo apt-get update -y
!sudo apt-get install ngrok -y


!ngrok version

deb https://ngrok-agent.s3.amazonaws.com buster main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://ngrok-agent.s3.amazonaws.com buster InRelease [20.3 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ngrok-agent.s3.amazonaws.com buster/main amd64 Packages [11.6 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]

## Cell 6 — WebSocket server + ngrok

In [21]:
import os
test = os.path.join(WORK_DIR, 'VCTK-corpus/p228/p228_023.wav')
print(os.path.exists(test), test)

False /content/drive/MyDrive/Github1/Accent-Conversion/VCTK-corpus/p228/p228_023.wav


In [22]:
# ==============================================================
# STT (Whisper) setup
# ==============================================================
from faster_whisper import WhisperModel
import tempfile
import os
import soundfile as sf

STT_MODEL_SIZE = "medium"
STT_LOCAL_MODEL_DIR = os.path.join(WORK_DIR, "offline_whisper_model")
STT_DEVICE = "cuda" if device.type == "cuda" else "cpu"
STT_COMPUTE_TYPE = "float16" if device.type == "cuda" else "int8"

print("=" * 60)
print("🗣️ Initializing STT model...")
print(f"STT model size: {STT_MODEL_SIZE}")
print(f"STT device: {STT_DEVICE}")
print(f"STT compute_type: {STT_COMPUTE_TYPE}")
print("=" * 60)

stt_model = WhisperModel(
    STT_MODEL_SIZE,
    device=STT_DEVICE,
    compute_type=STT_COMPUTE_TYPE,
    download_root=STT_LOCAL_MODEL_DIR
)

def _segments_to_text(segs):
    return " ".join((s.text or "").strip() for s in segs).strip()

def stt_detect_transcribe_translate(wav_np, sample_rate=24000):
    """
    Target behavior:
    1) Detect language + transcribe original speech
    2) If English:
         - keep English transcript
         - English translation = same as transcript
    3) If non-English:
         - keep original-language transcript
         - also translate into English

    Returns:
        {
            "language": ...,
            "language_probability": ...,
            "is_english": True/False,
            "transcribed_text": ...,
            "translated_text_en": ...,
            "segments": [...],
            "translated_segments_en": [...],
            "message": ...
        }
    """
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp_path = tmp.name

        sf.write(tmp_path, wav_np, sample_rate)

        # ----------------------------------------------------------
        # Step 1: detect language + original transcription
        # ----------------------------------------------------------
        segments_transcribe, info = stt_model.transcribe(
            tmp_path,
            beam_size=5,
            task="transcribe"
        )
        transcribe_segs = list(segments_transcribe)

        detected_language = getattr(info, "language", None)
        detected_prob = float(getattr(info, "language_probability", 0.0) or 0.0)
        is_english = str(detected_language).lower() in ["en", "english"]

        transcribed_text = _segments_to_text(transcribe_segs)

        # ----------------------------------------------------------
        # Step 2: if English -> keep transcript directly
        # ----------------------------------------------------------
        if is_english:
            return {
                "language": detected_language,
                "language_probability": detected_prob,
                "is_english": True,
                "transcribed_text": transcribed_text,
                "translated_text_en": transcribed_text,
                "segments": [
                    {
                        "start": float(s.start),
                        "end": float(s.end),
                        "text": (s.text or "").strip()
                    }
                    for s in transcribe_segs
                ],
                "translated_segments_en": [
                    {
                        "start": float(s.start),
                        "end": float(s.end),
                        "text": (s.text or "").strip()
                    }
                    for s in transcribe_segs
                ],
                "message": "English detected. Transcript returned directly."
            }

        # ----------------------------------------------------------
        # Step 3: if non-English -> translate to English
        # ----------------------------------------------------------
        segments_translate, info_translate = stt_model.transcribe(
            tmp_path,
            beam_size=5,
            task="translate"
        )
        translate_segs = list(segments_translate)
        translated_text_en = _segments_to_text(translate_segs)

        return {
            "language": detected_language,
            "language_probability": detected_prob,
            "is_english": False,
            "transcribed_text": transcribed_text,
            "translated_text_en": translated_text_en,
            "segments": [
                {
                    "start": float(s.start),
                    "end": float(s.end),
                    "text": (s.text or "").strip()
                }
                for s in transcribe_segs
            ],
            "translated_segments_en": [
                {
                    "start": float(s.start),
                    "end": float(s.end),
                    "text": (s.text or "").strip()
                }
                for s in translate_segs
            ],
            "message": "Non-English detected. Original transcript and English translation returned."
        }

    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)

print("✅ STT model ready.")

🗣️ Initializing STT model...
STT model size: medium
STT device: cuda
STT compute_type: float16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ STT model ready.


In [ ]:
import asyncio, json, io, base64, os, subprocess, shutil
import soundfile as sf
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import FileResponse, JSONResponse, HTMLResponse
from pyngrok import ngrok, conf
import nest_asyncio
import uvicorn
from IPython.display import display, Javascript # 🌟 Imported for auto-opening tabs

app = FastAPI()

print('=' * 60)
print('🚀 Building server (Backend + Frontend Hosting)...')
print('=' * 60)

# ------------------------------------------------------------
# Required objects from earlier notebook cells
# ------------------------------------------------------------
required_names = [
    'ensure_preloaded_refs',
    'DEMO_REFS',
    'decode_webm_to_pcm',
    'is_silence',
    'run_inference',
    'stt_detect_transcribe_translate',
    'SAMPLE_RATE',
    'SERVER_PORT',
    'NGROK_AUTH_TOKEN',
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required notebook variables/functions before server cell: {missing}")

# ------------------------------------------------------------
# Preload reference speakers
# ------------------------------------------------------------
PRELOADED_REFS = ensure_preloaded_refs()
DEFAULT_SPEAKERS = list(PRELOADED_REFS.keys())
print('Loaded speakers:', DEFAULT_SPEAKERS)

def wav_to_base64(wav_np, sample_rate=24000):
    buf = io.BytesIO()
    sf.write(buf, wav_np, sample_rate, format='WAV')
    return base64.b64encode(buf.getvalue()).decode('utf-8')

# Global variable to store the generated WebSocket URL
GLOBAL_WS_URL = ""

# 🌟 Frontend hosting and automated script injection route
@app.get("/")
async def serve_frontend():
    # Try finding the HTML file in several possible paths
    html_filename = 'Accent_Cat.html'
    candidates = [
        html_filename,
        os.path.join("Demo", html_filename),
        os.path.join("UI", html_filename),
        os.path.join("/content/drive/MyDrive/Github/Accent-Conversion/Demo", html_filename)
    ]

    html_path = None
    for c in candidates:
        if os.path.exists(c):
            html_path = c
            break

    if html_path is None:
        return HTMLResponse(
            content=f"<h1>Error: Frontend file '{html_filename}' not found</h1><p>Please ensure the file is uploaded to Colab or is in the correct Demo/UI folder.</p>",
            status_code=404
        )

    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    # Inject automated script: fill in WS link and auto-click connect
    inject_script = f"""
    <script>
        window.addEventListener('DOMContentLoaded', (event) => {{
            const wsInput = document.getElementById('wsUrl');
            if(wsInput) {{
                wsInput.value = '{GLOBAL_WS_URL}';
            }}
            const connectBtn = document.getElementById('connectBtn');
            if(connectBtn) {{
                setTimeout(() => {{
                    connectBtn.click();
                    console.log("Auto-connected to: " + '{GLOBAL_WS_URL}');
                }}, 600); // 600ms delay to ensure UI rendering is complete
            }}
        }});
    </script>
    </body>
    """
    html_content = html_content.replace('</body>', inject_script)
    return HTMLResponse(content=html_content)


@app.get("/ref/{speaker}")
async def get_reference_audio(speaker: str):
    speaker = str(speaker).strip()

    if speaker not in DEMO_REFS:
        return JSONResponse(
            status_code=404,
            content={"error": f"Speaker {speaker} not found in DEMO_REFS."}
        )

    rel_path = DEMO_REFS[speaker]

    wav_path = None
    if 'resolve_demo_ref_path' in globals():
        wav_path = resolve_demo_ref_path(rel_path)

    if wav_path is None or not os.path.exists(wav_path):
        candidates = [rel_path]
        if 'WORK_DIR' in globals():
            candidates += [
                os.path.join(WORK_DIR, rel_path),
                os.path.join(WORK_DIR, "Demo", rel_path),
            ]
        candidates += [os.path.join("Demo", rel_path)]
        for c in candidates:
            if os.path.exists(c):
                wav_path = c
                break

    if wav_path is None or not os.path.exists(wav_path):
        return JSONResponse(
            status_code=404,
            content={"error": f"Reference audio for {speaker} not found.", "rel_path": rel_path}
        )

    print(f"[REF] Serving preview for {speaker}: {wav_path}")
    return FileResponse(wav_path, media_type="audio/wav", filename=f"{speaker}.wav")


@app.websocket('/ws/vc')
async def ws_endpoint(websocket: WebSocket):
    await websocket.accept()
    print('[WS] 🟢 Client connected.')

    expect_audio_upload = False
    cached_source_samples = None
    cached_source_stt = None
    converted_cache = {}

    try:
        while True:
            message = await websocket.receive()

            if 'bytes' in message and message['bytes'] is not None:
                if not expect_audio_upload:
                    print('[WS] ⚠️ Unexpected binary (ignoring)')
                    continue

                expect_audio_upload = False
                data = message['bytes']
                print(f'[WS] ▶️ Received recorded audio ({len(data)} bytes)')

                try:
                    samples = decode_webm_to_pcm(data)
                except Exception as e:
                    await websocket.send_text(json.dumps({
                        'status': 'error',
                        'message': f'Audio decode failed: {e}'
                    }))
                    continue

                if is_silence(samples):
                    await websocket.send_text(json.dumps({'status': 'silence'}))
                    continue

                cached_source_samples = samples
                converted_cache = {}

                try:
                    cached_source_stt = stt_detect_transcribe_translate(samples, SAMPLE_RATE)
                except Exception as e:
                    cached_source_stt = {
                        "language": None,
                        "transcribed_text": "",
                        "translated_text_en": "",
                        "message": f"STT failed: {e}"
                    }

                await websocket.send_text(json.dumps({
                    'status': 'source_ready',
                    'seconds': round(len(samples) / SAMPLE_RATE, 2),
                    'source_stt': cached_source_stt
                }))
                continue

            if 'text' in message and message['text'] is not None:
                try:
                    msg = json.loads(message['text'])
                except Exception:
                    continue

                cmd = msg.get('cmd', '')

                if cmd == 'upload_audio':
                    expect_audio_upload = True
                    await websocket.send_text(json.dumps({'status': 'ready_for_audio'}))

                elif cmd == 'get_source_stt':
                    if cached_source_samples is None:
                        await websocket.send_text(json.dumps({'status': 'error', 'message': 'No recorded source audio uploaded yet.'}))
                    else:
                        await websocket.send_text(json.dumps({'status': 'source_stt', 'source_stt': cached_source_stt}))

                elif cmd == 'convert_speaker':
                    speaker = str(msg.get('speaker', '')).strip()

                    if cached_source_samples is None:
                        await websocket.send_text(json.dumps({'status': 'error', 'message': 'No recorded source audio uploaded yet.'}))
                        continue

                    if speaker not in PRELOADED_REFS:
                        await websocket.send_text(json.dumps({'status': 'error', 'message': f'Speaker {speaker} not available.'}))
                        continue

                    if speaker in converted_cache:
                        cached_item = converted_cache[speaker]
                        await websocket.send_text(json.dumps({
                            'status': 'speaker_done',
                            'speaker': speaker,
                            'audio_base64': cached_item['audio_base64'],
                            'cached': True
                        }))
                        continue

                    ref_mel, domain_idx = PRELOADED_REFS[speaker]
                    print(f'[WS] ⚙️ Converting cached source -> {speaker} (domain={domain_idx})')

                    try:
                        converted = run_inference(cached_source_samples, ref_mel, domain_idx)
                    except Exception as e:
                        await websocket.send_text(json.dumps({'status': 'error', 'message': f'Inference error: {e}'}))
                        continue

                    audio_b64 = wav_to_base64(converted, SAMPLE_RATE)
                    converted_cache[speaker] = {'audio_base64': audio_b64}

                    await websocket.send_text(json.dumps({
                        'status': 'speaker_done',
                        'speaker': speaker,
                        'audio_base64': audio_b64,
                        'cached': False
                    }))

                elif cmd == 'clear_source':
                    cached_source_samples = None
                    cached_source_stt = None
                    converted_cache = {}
                    await websocket.send_text(json.dumps({'status': 'source_cleared'}))

                elif cmd == 'list_speakers':
                    await websocket.send_text(json.dumps({'status': 'speakers', 'speakers': list(PRELOADED_REFS.keys())}))

    except WebSocketDisconnect:
        print('[WS] 🔴 Client disconnected.')
    except Exception:
        import traceback
        traceback.print_exc()

# ------------------------------------------
# Start ngrok + uvicorn
# ------------------------------------------
nest_asyncio.apply()

print('📦 Checking system ngrok...')
if not os.path.exists('/usr/bin/ngrok'):
    print('⏳ Installing ngrok via APT...')
    subprocess.run("curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null", shell=True)
    subprocess.run("echo 'deb https://ngrok-agent.s3.amazonaws.com buster main' | sudo tee /etc/apt/sources.list.d/ngrok.list", shell=True)
    subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -y -qq", shell=True)
    subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get install ngrok -y -qq", shell=True)

ngrok_path = '/usr/bin/ngrok' if os.path.exists('/usr/bin/ngrok') else shutil.which('ngrok')

if ngrok_path:
    os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTH_TOKEN
    my_config = conf.PyngrokConfig(ngrok_path=ngrok_path)

    try:
        ngrok.kill(pyngrok_config=my_config)
        kw = dict(pyngrok_config=my_config)
        if 'NGROK_DOMAIN' in globals() and NGROK_DOMAIN and str(NGROK_DOMAIN).strip():
            kw['domain'] = NGROK_DOMAIN

        tunnel = ngrok.connect(SERVER_PORT, **kw)

        # 🌟 Core: Calculate the URL and store it in a global variable for route use
        PUBLIC_URL = tunnel.public_url
        GLOBAL_WS_URL = PUBLIC_URL.replace('https://', 'wss://').replace('http://', 'ws://') + '/ws/vc'

        print('\n' + '=' * 70)
        print('🎉 All set! The UI should open automatically in a new tab.')
        print('⚠️ If your browser blocked the pop-up, please click the link below manually:')
        print(f'👉 {PUBLIC_URL}')
        print('=' * 70 + '\n')

        # 🌟 Trigger JavaScript to automatically open the URL in a new tab
        display(Javascript(f'window.open("{PUBLIC_URL}", "_blank");'))

    except Exception as e:
        print(f'❌ Tunnel error: {e}')

print('🚀 Starting Uvicorn...')
uvi_cfg = uvicorn.Config(app, host='0.0.0.0', port=SERVER_PORT, log_level='info')
server = uvicorn.Server(uvi_cfg)
await server.serve()

🚀 Building server (Backend + Frontend Hosting)...
Pre-loading reference mels for notebook demo...
  📎 p228: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p228/23.wav
  ✅ p228 (domain=1) mel=(1, 80, 437)
  📎 p230: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p230/23.wav
  ✅ p230 (domain=3) mel=(1, 80, 477)
  📎 p233: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p233/23.wav
  ✅ p233 (domain=5) mel=(1, 80, 487)
  📎 p236: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p236/23.wav
  ✅ p236 (domain=6) mel=(1, 80, 453)
  📎 p243: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p243/23.wav
  ✅ p243 (domain=13) mel=(1, 80, 468)
  📎 p244: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p244/23.wav
  ✅ p244 (domain=9) mel=(1, 80, 463)
  📎 p254: /content/drive/MyDrive/Github1/Accent-Conversion/Data/VCTK2019/p254/23.wav
  ✅ p254 (domain=14) mel=(1, 80, 449)
  📎 p258: /content/drive/MyDrive/Github1/Accen

<IPython.core.display.Javascript object>

INFO:     Started server process [36405]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Starting Uvicorn...
INFO:     86.40.118.19:0 - "GET / HTTP/1.1" 200 OK


INFO:     86.40.118.19:0 - "WebSocket /ws/vc" [accepted]
INFO:     connection open


[WS] 🟢 Client connected.
[WS] ▶️ Received recorded audio (44732 bytes)
  [VAD] RMS=0.06935  threshold=0.005
INFO:     86.40.118.19:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
[WS] ⚙️ Converting cached source -> p230 (domain=3)
[WS] ⚙️ Converting cached source -> p236 (domain=6)
[WS] ⚙️ Converting cached source -> p244 (domain=9)
[WS] ▶️ Received recorded audio (43766 bytes)
  [VAD] RMS=0.00346  threshold=0.005
[WS] ▶️ Received recorded audio (47630 bytes)
  [VAD] RMS=0.08070  threshold=0.005
[WS] ⚙️ Converting cached source -> p230 (domain=3)
[WS] ⚙️ Converting cached source -> p236 (domain=6)
[WS] ⚙️ Converting cached source -> p244 (domain=9)


Traceback (most recent call last):
  File "/tmp/ipykernel_36405/3348732674.py", line 150, in ws_endpoint
    message = await websocket.receive()
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/starlette/websockets.py", line 57, in receive
    raise RuntimeError('Cannot call "receive" once a disconnect message has been received.')
RuntimeError: Cannot call "receive" once a disconnect message has been received.
INFO:     connection closed
